# IonQ native cloud: QAOA + VQC from existing QASM files

Submit the **same** benchmark circuits used elsewhere in this repo to
**IonQ Quantum Cloud** (`cloud.ionq.com`) through the official
[`qiskit-ionq`](https://docs.ionq.com/sdks/qiskit) provider.

| Benchmark | QASM source | Format | Default shots |
|-----------|-------------|--------|---------------|
| **QAOA** (10-qubit ring MaxCut, p=2) | `QAOA_code/qaoa_v1.qasm` | OpenQASM 2.0 | 1024 |
| **VQC** (11 MNIST test images) | `data/circuits/qvc_test_*.qasm` | OpenQASM 3.0 | 4000 |

**Already in this repo**

- **VQC + XEB on IonQ native** (no QAOA): [`ionq-benchmarks-from-qasm.ipynb`](ionq-benchmarks-from-qasm.ipynb)
  — generated by `_build_ionq_benchmark_from_qasm_nb.py`.
- **QAOA on IonQ via Braket** (not native): `analysis/ionq_results.json` +
  `data_analysis.ipynb` entry `"ionq_Forte"`.
- **QAOA on Azure** (template, not native): `QAOA_code/azure-setup.py`.

**Credentials**

Get an API key from https://cloud.ionq.com/settings/keys and either:

- `export IONQ_API_KEY=...` before starting the kernel, **or**
- paste the key into `IONQ_API_KEY` in the config cell below.

**Recommended order**

1. `simulator` (ideal) — smoke test, free
2. `simulator` + `noise_model="forte-1"` — free noise model
3. `qpu.forte-1` — real QPU (uses credits; queue may take hours)

Set `VQC_TEST_LIMIT = 2` for a cheap first VQC run.


In [ ]:
# Run once per fresh kernel.
%pip install -q "qiskit>=1.2,<2" qiskit-qasm3-import "qiskit-ionq<1" scipy matplotlib


In [ ]:
from __future__ import annotations

import csv
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile, qasm2, qasm3
from qiskit.transpiler.passes import RemoveBarriers
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%SZ")
OUTPUT_ROOT = Path("../data/raw/ionq_native") / RUN_STAMP
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Results will be written under:", OUTPUT_ROOT.resolve())


## Configuration — edit this cell only


In [ ]:
# Which benchmarks to run on each backend listed below.
RUN_QAOA = True
RUN_VQC = True

# IonQ backends — same schema as ionq-benchmarks-from-qasm.ipynb
IONQ_DEVICES: List[Dict[str, Any]] = [
    {"backend": "simulator", "slug": "simulator_ideal"},
    # {"backend": "simulator", "noise_model": "forte-1", "slug": "simulator_forte_1"},
    # {"backend": "qpu.forte-1", "slug": "qpu_forte_1"},
]

IONQ_API_KEY: Optional[str] = None  # or export IONQ_API_KEY

SHOTS_QAOA = 1024
SHOTS_VQC = 4000
VQC_TEST_LIMIT: Optional[int] = None  # e.g. 2 for smoke test
IONQ_BATCH_SIZE: Optional[int] = 25
TRANSPILE_OPTLEVEL = 1
NUM_QUBITS = 10
IONQ_JOB_TIMEOUT_S: Optional[float] = None
IONQ_POLL_S = 5.0

# Paths — leave None for auto-discovery relative to repo layout.
QAOA_QASM_PATH: Optional[Path] = None
VQC_QASM_DIR: Optional[Path] = None


## Connect to IonQ Quantum Cloud


In [ ]:
def _resolve_ionq_api_key() -> str:
    explicit = globals().get("IONQ_API_KEY")
    if explicit:
        return str(explicit)
    for var in ("IONQ_API_KEY", "IONQ_API_TOKEN"):
        v = os.environ.get(var)
        if v:
            return v
    raise RuntimeError(
        "No IonQ API key. Set IONQ_API_KEY in the config cell or export it in your shell."
    )


from qiskit_ionq import IonQProvider  # noqa: E402

_PROVIDER = IonQProvider(_resolve_ionq_api_key())
print("IonQ provider initialised.")

try:
    names = sorted({b.name for b in _PROVIDER.backends()})
    print(f"{len(names)} visible backend(s) — copy one into IONQ_DEVICES:")
    for n in names:
        print(" ", repr(n))
except Exception as exc:
    print("Could not list backends:", exc)


## Helpers — paths, QASM load, IonQ submit


In [ ]:
def _resolve_qaoa_qasm_path() -> Path:
    explicit = globals().get("QAOA_QASM_PATH")
    if explicit is not None:
        p = Path(explicit).expanduser().resolve()
        if not p.is_file():
            raise FileNotFoundError(p)
        return p
    here = Path.cwd().resolve()
    candidates = [
        here / "QAOA_code" / "qaoa_v1.qasm",
        here.parent / "QAOA_code" / "qaoa_v1.qasm",
        here / "qaoa_v1.qasm",
    ]
    for c in candidates:
        if c.is_file():
            return c.resolve()
    raise FileNotFoundError(
        "Could not find QAOA_code/qaoa_v1.qasm — set QAOA_QASM_PATH explicitly."
    )


def _resolve_vqc_qasm_dir() -> Path:
    explicit = globals().get("VQC_QASM_DIR")
    if explicit is not None:
        p = Path(explicit).expanduser().resolve()
        if not p.is_dir():
            raise FileNotFoundError(p)
        return p
    here = Path.cwd().resolve()
    candidates = [
        here.parent / "data" / "circuits",
        here.parent / "data" / "circuits",
        here.parent / "data" / "circuits",
    ]
    for c in candidates:
        if c.is_dir():
            return c.resolve()
    raise FileNotFoundError(
        "Could not find data/circuits/ — set VQC_QASM_DIR explicitly."
    )


_VQC_FNAME_RE = re.compile(r"qvc_test_(\d+)\.qasm$")
_VQC_LABEL_RE = re.compile(r"true\s+label\s+(-?\d+)", re.IGNORECASE)
_REMOVE_BARRIERS = RemoveBarriers()


def _parse_vqc_index(path: Path) -> int:
    m = _VQC_FNAME_RE.search(path.name)
    if not m:
        raise ValueError(f"Not a VQC test QASM file: {path.name}")
    return int(m.group(1))


def _parse_vqc_label(path: Path) -> int:
    for line in path.read_text().splitlines()[:5]:
        m = _VQC_LABEL_RE.search(line)
        if m:
            return int(m.group(1))
    raise ValueError(f"No 'true label' in header of {path.name}")


def load_qaoa_circuit(path: Path) -> QuantumCircuit:
    qc = qasm2.load(str(path))
    if qc.num_qubits != NUM_QUBITS:
        raise ValueError(f"{path.name}: expected {NUM_QUBITS} qubits, got {qc.num_qubits}")
    return _REMOVE_BARRIERS(qc)


def load_vqc_circuit(path: Path) -> QuantumCircuit:
    qc = qasm3.load(str(path))
    if qc.num_qubits != NUM_QUBITS:
        raise ValueError(f"{path.name}: expected {NUM_QUBITS} qubits, got {qc.num_qubits}")
    return _REMOVE_BARRIERS(qc)


def _normalise_counts(counts: Dict[str, Any], shots: int) -> Dict[str, int]:
    if not counts:
        return {}
    total = sum(counts.values())
    if total <= 1.5:
        return {k.replace(" ", ""): int(round(v * shots)) for k, v in counts.items()}
    return {k.replace(" ", ""): int(v) for k, v in counts.items()}


def parity_expectation_from_counts(counts: Dict[str, int]) -> float:
    total = sum(counts.values())
    if total <= 0:
        return 0.0
    s = 0.0
    for b, n in counts.items():
        parity = b.count("1") % 2
        s += n * ((-1.0) ** parity)
    return s / total


def _device_slug(cfg: Dict[str, Any]) -> str:
    if cfg.get("slug"):
        return str(cfg["slug"])
    base = str(cfg["backend"])
    nm = cfg.get("noise_model")
    candidate = f"{base}__{nm}" if nm else base
    s = re.sub(r"[^0-9A-Za-z._-]+", "_", candidate)
    return s[:120] if len(s) > 120 else s


def _get_ionq_backend(cfg: Dict[str, Any]):
    name = cfg["backend"]
    backend = _PROVIDER.get_backend(name)
    nm = cfg.get("noise_model")
    if nm is not None:
        if not name.endswith("simulator"):
            raise ValueError(f"noise_model only valid on simulator, got {name!r}")
        backend.set_options(noise_model=nm)
    return backend


def _ionq_backend_name(backend) -> str:
    # Qiskit BackendV1 exposes .name() as a method; BackendV2 uses a property.
    name = getattr(backend, "name", None)
    if callable(name):
        return str(name())
    if name is not None:
        return str(name)
    return str(backend)


def _transpile_for_ionq(backend, qc: QuantumCircuit) -> QuantumCircuit:
    # Transpile for IonQ across qiskit-ionq versions. Older simulator backends
    # lack ``target``; IonQ docs recommend transpile(..., backend=backend).
    target = getattr(backend, "target", None)
    if target is not None:
        pm = generate_preset_pass_manager(
            target=target, optimization_level=TRANSPILE_OPTLEVEL,
        )
        return pm.run(qc)
    return transpile(qc, backend=backend, optimization_level=TRANSPILE_OPTLEVEL)


def _is_outcome_key(key: Any) -> bool:
    s = str(key)
    if "-" in s and len(s) >= 32:
        return False
    try:
        int(s)
        return True
    except ValueError:
        return False


def _is_leaf_histogram(obj: Any) -> bool:
    return (
        isinstance(obj, dict)
        and bool(obj)
        and all(_is_outcome_key(k) and isinstance(v, (int, float)) for k, v in obj.items())
    )


def _split_ionq_histogram_payload(raw: Any) -> List[dict]:
    if isinstance(raw, list):
        return [h for h in raw if isinstance(h, dict)]
    if not isinstance(raw, dict) or not raw:
        return []
    if _is_leaf_histogram(raw):
        return [raw]
    if all(isinstance(v, dict) for v in raw.values()):
        inner = list(raw.values())
        if inner and all(_is_leaf_histogram(h) for h in inner):
            return inner
    raise ValueError(
        "Unrecognized IonQ results payload. "
        f"top-level keys sample: {list(raw.keys())[:3] if isinstance(raw, dict) else raw}"
    )


def _ionq_hist_to_bitstring_counts(
    hist: dict,
    num_qubits: int,
    clbits: List[int],
    shots: int,
) -> Dict[str, int]:
    def get_bitvalue(bitstring: str, bit: int) -> str:
        if bit is not None and 0 <= bit < len(bitstring):
            return bitstring[bit]
        return "0"

    mapped: Dict[int, float] = {}
    for value, weight in hist.items():
        if not _is_outcome_key(value):
            continue
        bitstring = bin(int(value))[2:].rjust(num_qubits, "0")[::-1]
        outvalue = int("".join(get_bitvalue(bitstring, bit) for bit in clbits)[::-1], 2)
        mapped[outvalue] = mapped.get(outvalue, 0.0) + float(weight)

    total = sum(mapped.values())
    is_prob = total <= 1.5
    counts: Dict[str, int] = {}
    for key, val in mapped.items():
        bits = bin(int(key))[2:].rjust(num_qubits, "0")
        count = int(round(val * shots)) if is_prob else int(round(val))
        if count:
            counts[bits] = count
    return counts


def _fetch_chunk_counts_ionq(job, chunk: List[QuantumCircuit], shots: int) -> List[Dict[str, int]]:
    job.status()
    if not getattr(job, "_results_url", None):
        raise RuntimeError(f"job {job.job_id()}: missing results URL after completion")
    raw = job._client.get_results(job._results_url)
    histograms = _split_ionq_histogram_payload(raw)
    num_qubits = getattr(job, "_num_qubits", None) or NUM_QUBITS
    clbits_list = getattr(job, "_clbits", None) or [list(range(num_qubits))] * len(chunk)
    if len(clbits_list) == 1 and len(chunk) > 1:
        clbits_list = clbits_list * len(chunk)
    if len(histograms) != len(chunk):
        raise RuntimeError(
            f"job {job.job_id()}: expected {len(chunk)} histograms, got {len(histograms)}"
        )
    return [
        _ionq_hist_to_bitstring_counts(histograms[i], num_qubits, clbits_list[i], shots)
        for i in range(len(chunk))
    ]


def _submit_chunks(
    backend,
    circuits: List[QuantumCircuit],
    shots: int,
    batch_size: Optional[int],
) -> Tuple[List[Dict[str, int]], List[str]]:
    if not circuits:
        return [], []
    n = len(circuits)
    bs = batch_size if (batch_size and batch_size > 0) else n
    counts_all: List[Dict[str, int]] = []
    job_ids: List[str] = []
    for start in range(0, n, bs):
        chunk = circuits[start : start + bs]
        job = backend.run(chunk, shots=shots)
        jid = job.job_id()
        job_ids.append(jid)
        print(f"    submitted {len(chunk)} circuit(s) -> job_id={jid}")
        job.wait_for_final_state(timeout=IONQ_JOB_TIMEOUT_S, wait=IONQ_POLL_S)
        chunk_counts = _fetch_chunk_counts_ionq(job, chunk, shots)
        if len(chunk_counts) != len(chunk):
            raise RuntimeError(
                f"Job {jid}: expected {len(chunk)} count dicts, got {len(chunk_counts)}"
            )
        counts_all.extend(chunk_counts)
    return counts_all, job_ids


## Run QAOA


In [ ]:
QAOA_METADATA = {
    "num_qubits": 10,
    "num_layers": 2,
    "gamma": [0.2052, 0.3974],
    "beta": [0.1026, 0.2948],
    "graph": "10-vertex ring",
}


def run_qaoa_ionq(cfg: Dict[str, Any], out_dir: Path) -> Dict[str, Any]:
    out_dir.mkdir(parents=True, exist_ok=True)
    qasm_path = _resolve_qaoa_qasm_path()
    backend = _get_ionq_backend(cfg)
    qc = load_qaoa_circuit(qasm_path)
    compiled = _transpile_for_ionq(backend, qc)
    print(f"  [QAOA] {qasm_path.name} -> {_ionq_backend_name(backend)}  ({SHOTS_QAOA} shots)")

    t0 = time.time()
    counts_list, job_ids = _submit_chunks(
        backend, [compiled], shots=SHOTS_QAOA, batch_size=IONQ_BATCH_SIZE,
    )
    counts = counts_list[0]
    dt = time.time() - t0

    (out_dir / "job_ids.txt").write_text("\n".join(job_ids) + "\n")

    payload = {
        "run_stamp": RUN_STAMP,
        "provider": "ionq_native",
        "backend": _ionq_backend_name(backend),
        "backend_name_arg": cfg["backend"],
        "noise_model": cfg.get("noise_model"),
        "ionq_job_ids": job_ids,
        "qasm_file": str(qasm_path),
        "shots": SHOTS_QAOA,
        "total_shots_returned": sum(counts.values()),
        "num_unique_bitstrings": len(counts),
        "qaoa_parameters": QAOA_METADATA,
        "measurement_counts": counts,
        "measuredQubits": list(range(NUM_QUBITS)),
        "wall_time_s": dt,
        "utc_finished": datetime.now(timezone.utc).isoformat(),
    }
    out_path = out_dir / "qaoa_results.json"
    out_path.write_text(json.dumps(payload, indent=2))
    print(f"  [QAOA] done in {dt:.1f}s -> {out_path}")
    return payload


## Run VQC


In [ ]:
def run_vqc_ionq(cfg: Dict[str, Any], out_dir: Path) -> Dict[str, Any]:
    out_dir.mkdir(parents=True, exist_ok=True)
    qasm_dir = _resolve_vqc_qasm_dir()
    backend = _get_ionq_backend(cfg)

    files = sorted(qasm_dir.glob("qvc_test_*.qasm"), key=_parse_vqc_index)
    if not files:
        raise FileNotFoundError(f"No qvc_test_*.qasm under {qasm_dir}")
    if VQC_TEST_LIMIT is not None:
        files = files[:VQC_TEST_LIMIT]
    print(f"  [VQC] {len(files)} circuit(s) from {qasm_dir} -> {_ionq_backend_name(backend)}")

    circuits: List[QuantumCircuit] = []
    meta: List[Dict[str, Any]] = []
    for path in files:
        idx = _parse_vqc_index(path)
        label = _parse_vqc_label(path)
        circuits.append(_transpile_for_ionq(backend, load_vqc_circuit(path)))
        meta.append({"index": idx, "true_label": label, "qasm_file": path.name})

    t0 = time.time()
    counts_list, job_ids = _submit_chunks(
        backend, circuits, shots=SHOTS_VQC, batch_size=IONQ_BATCH_SIZE,
    )
    dt = time.time() - t0

    (out_dir / "job_ids.txt").write_text("\n".join(job_ids) + "\n")
    counts_dir = out_dir / "counts"
    counts_dir.mkdir(exist_ok=True)

    rows = []
    for m, cdict in zip(meta, counts_list):
        idx = m["index"]
        (counts_dir / f"qvc_test_{idx:03d}.json").write_text(json.dumps(cdict))
        ev = parity_expectation_from_counts(cdict)
        pred = 1 if ev >= 0 else -1
        rows.append({**m, "ev": ev, "pred": pred})

    y_true = np.array([r["true_label"] for r in rows], dtype=int)
    y_pred = np.array([r["pred"] for r in rows], dtype=int)
    summary = {
        "benchmark": "vqc_inference_from_qasm",
        "provider": "ionq_native",
        "backend": _ionq_backend_name(backend),
        "noise_model": cfg.get("noise_model"),
        "qasm_dir": str(qasm_dir),
        "shots": SHOTS_VQC,
        "n_images": len(rows),
        "accuracy": float(np.mean(y_true == y_pred)),
        "mean_abs_parity_ev": float(np.mean([abs(r["ev"]) for r in rows])),
        "wall_time_s": dt,
        "ionq_job_ids": job_ids,
        "utc_finished": datetime.now(timezone.utc).isoformat(),
    }
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))

    with (out_dir / "predictions.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["index", "true_label", "qasm_file", "ev", "pred"])
        w.writeheader()
        for r in rows:
            w.writerow(r)

    print(
        f"  [VQC] accuracy={summary['accuracy']*100:.2f}%  "
        f"mean|<Z^10>|={summary['mean_abs_parity_ev']:.4f}  "
        f"time={dt:.1f}s -> {out_dir}"
    )
    return summary


## Sanity-check QASM paths


In [ ]:
print("QAOA QASM:", _resolve_qaoa_qasm_path())
print("VQC QASM dir:", _resolve_vqc_qasm_dir())

vqc_dir = _resolve_vqc_qasm_dir()
vqc_files = sorted(vqc_dir.glob("qvc_test_*.qasm"), key=_parse_vqc_index)
print(f"Found {len(vqc_files)} VQC test file(s)")
for p in vqc_files[:3]:
    print(f"  {p.name}: label={_parse_vqc_label(p)}")

qc = load_qaoa_circuit(_resolve_qaoa_qasm_path())
print(f"QAOA circuit: {qc.num_qubits} qubits, depth {qc.depth()}")


## Submit jobs


In [ ]:
def run_device(cfg: Dict[str, Any]) -> Path:
    slug = _device_slug(cfg)
    out = OUTPUT_ROOT / slug
    tag = cfg["backend"] + (f" + noise={cfg['noise_model']}" if cfg.get("noise_model") else "")
    print(f"\n=== {tag} (slug={slug}) ===")

    index: Dict[str, str] = {}
    if RUN_QAOA:
        run_qaoa_ionq(cfg, out / "qaoa")
        index["qaoa"] = str((out / "qaoa" / "qaoa_results.json").resolve())
    if RUN_VQC:
        run_vqc_ionq(cfg, out / "vqc")
        index["vqc"] = str((out / "vqc" / "summary.json").resolve())

    (out / "device_index.json").write_text(json.dumps(index, indent=2))
    print(f"=== finished {tag} -> {out.resolve()} ===\n")
    return out


if not IONQ_DEVICES:
    raise ValueError("IONQ_DEVICES is empty — add at least one backend.")

master: Dict[str, str] = {}
for cfg in IONQ_DEVICES:
    master[_device_slug(cfg)] = str(run_device(cfg).resolve())
(OUTPUT_ROOT / "all_devices_index.json").write_text(json.dumps(master, indent=2))
print("Master index:", (OUTPUT_ROOT / "all_devices_index.json").resolve())


## Recover results from a completed IonQ job

Use this if **Submit jobs** crashed after IonQ finished (e.g. JSON save bug) but
the job shows as completed in your IonQ dashboard.


In [ ]:
# Example: recover the QAOA job that finished but failed to save locally.
RETRIEVE_JOB_ID = "019f3bb5-d05e-712b-8587-ab239fe16fdc"  # paste your job id
RETRIEVE_BACKEND = "simulator"
RETRIEVE_SHOTS = SHOTS_QAOA
RECOVER_OUT = OUTPUT_ROOT / "simulator_ideal" / "qaoa_recovered"
RECOVER_OUT.mkdir(parents=True, exist_ok=True)

backend = _get_ionq_backend({"backend": RETRIEVE_BACKEND})
job = backend.retrieve_job(RETRIEVE_JOB_ID)
print("status:", job.status())
job.wait_for_final_state(timeout=IONQ_JOB_TIMEOUT_S, wait=IONQ_POLL_S)

raw = job.get_counts()
counts = _normalise_counts(raw if isinstance(raw, dict) else raw[0], RETRIEVE_SHOTS)

payload = {
    "run_stamp": RUN_STAMP,
    "provider": "ionq_native",
    "backend": _ionq_backend_name(backend),
    "backend_name_arg": RETRIEVE_BACKEND,
    "ionq_job_ids": [RETRIEVE_JOB_ID],
    "qasm_file": str(_resolve_qaoa_qasm_path()),
    "shots": RETRIEVE_SHOTS,
    "total_shots_returned": sum(counts.values()),
    "num_unique_bitstrings": len(counts),
    "qaoa_parameters": QAOA_METADATA,
    "measurement_counts": counts,
    "measuredQubits": list(range(NUM_QUBITS)),
    "recovered_from_job_id": RETRIEVE_JOB_ID,
    "utc_recovered": datetime.now(timezone.utc).isoformat(),
}
out_path = RECOVER_OUT / "qaoa_results.json"
out_path.write_text(json.dumps(payload, indent=2))
print("Wrote", out_path.resolve())
print("top counts:", sorted(counts.items(), key=lambda x: -x[1])[:5])


## Plug results into `data_analysis.ipynb`

**QAOA:** copy `.../qaoa/qaoa_results.json` and add to `RESULT_FILES` or
`IQM_BRACKET_FILES`-style loader in `analysis/data_analysis.ipynb`
(key `"measurement_counts"` is already supported).

**VQC:** copy counts into a flat folder and follow the pattern in
`vqc_braket_*_analysis.ipynb`, or extend `vqc_multi_device_comparison.ipynb`
`DEVICE_REGISTRY`.
